# Assignment 2 - Experiment Tracking
Jugaad Singh Sohal (MDS202421)
## Data Preparation with DVC

Prepare the SMS spam data and use DVC (Data Version Control) to track different versions of our data splits.

### Installing required packages

In [1]:
%pip install pandas numpy scikit-learn dvc dvc-gdrive


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

### Download the dataset

UCI SMS Spam collection dataset

In [3]:
!mkdir -p data
!wget -q https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip -O data/smsspamcollection.zip
!unzip -o data/smsspamcollection.zip -d data/
!ls data/

Archive:  data/smsspamcollection.zip
  inflating: data/SMSSpamCollection  
  inflating: data/readme             


raw_data.csv	  readme	     smsspamcollection.zip
raw_data.csv.dvc  SMSSpamCollection


### Load and save raw data

In [4]:
df = pd.read_csv('data/SMSSpamCollection', sep='\t', header=None, names=['label', 'message'], encoding='latin-1')
print(f"Total messages: {len(df)}")
print(f"\nClass distribution:")
print(df['label'].value_counts())
print(f"\nSpam percentage: {(df['label'] == 'spam').mean() * 100:.2f}%")

df.to_csv('data/raw_data.csv', index=False)
print(f"\nSaved raw_data.csv ({len(df)} rows)")
df.head()

Total messages: 5572

Class distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

Spam percentage: 13.41%

Saved raw_data.csv (5572 rows)


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Initialize DVC

DVC (Data Version Control) tracks data files alongside git. `dvc init --subdir` initializes DVC inside a subdirectory of the git repository.

In [ ]:
# !dvc init --subdir

ERROR: failed to initiate DVC - '.dvc' exists. Use `-f` to force.


### Track raw data with DVC

`dvc add` replaces the actual data file with a `.dvc` pointer file in git, and caches the data in `.dvc/cache/`.

In [6]:
!dvc add data/raw_data.csv


!
 


⠋ Checking graph


!
  0% Adding...|                                      |0/1 [00:00<?,     ?file/s]
Adding...                                                                       

!




                                                                                

!

  0% Checking cache in '/home/jugaad/github/appliedmachinelearning/Assignment-02

                                                                                

!

  0%|          |Checking out /home/jugaad/github/appli0/1 [00:00<?,    ?files/s]



                                                                                
100% Adding...|████████████████████████████████████████|1/1 [00:00, 13.85file/s]

To track the changes with git, run:

	git add data/raw_data.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


### Version 1: Train/Validation/Test Split (random_state=42)

Stratified 70/15/15 split to maintain the same spam/ham ratio in each set.

In [7]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

train_df[['label', 'message']].to_csv('data/train.csv', index=False)
val_df[['label', 'message']].to_csv('data/validation.csv', index=False)
test_df[['label', 'message']].to_csv('data/test.csv', index=False)

print(f"Train: {len(train_df)} samples")
print(f"Validation: {len(val_df)} samples")
print(f"Test: {len(test_df)} samples")

print(f"\nTrain distribution:")
print(f"  ham:  {(train_df['label'] == 'ham').sum()}")
print(f"  spam: {(train_df['label'] == 'spam').sum()}")
print(f"\nValidation distribution:")
print(f"  ham:  {(val_df['label'] == 'ham').sum()}")
print(f"  spam: {(val_df['label'] == 'spam').sum()}")
print(f"\nTest distribution:")
print(f"  ham:  {(test_df['label'] == 'ham').sum()}")
print(f"  spam: {(test_df['label'] == 'spam').sum()}")

Train: 3900 samples
Validation: 836 samples
Test: 836 samples

Train distribution:
  ham:  3377
  spam: 523

Validation distribution:
  ham:  724
  spam: 112

Test distribution:
  ham:  724
  spam: 112


In [8]:
!dvc add data/train.csv data/validation.csv data/test.csv


!
 


⠋ Checking graph


!
  0% Adding...|                                      |0/3 [00:00<?,     ?file/s]
  0% Adding...|                     | data/train.csv |0/3 [00:00<?,     ?file/s]

!




                                                                                

!

  0% Checking cache in '/home/jugaad/github/appliedmachinelearning/Assignment-02

                                                                                

!

  0%|          |Adding data/train.csv to cache        0/1 [00:00<?,     ?file/s]

                                                                                

!

  0%|          |Checking out /home/jugaad/github/appli0/1 [00:00<?,    ?files/s]

                                                                                


  0% Adding...|                | data/validation.csv |0/3 [00:00<?,     ?file/s]

!


                                                                                

!

  0% Checking cache in '/home/jugaad/github/appliedmachinelearning/Assignment-02

                                                                                

!

  0%|          |Adding data/validation.csv to cache   0/1 [00:00<?,     ?file/s]

                                                                                

!

  0%|          |Checking out /home/jugaad/github/appli0/1 [00:00<?,    ?files/s]

                                                                                
 67% Adding...|████████    | data/validation.csv |2/3 [00:00<00:00, 18.21file/s]
 67% Adding...|████████████      | data/test.csv |2/3 [00:00<00:00, 18.21file/s]

!


                                                                                

!

  0% Checking cache in '/home/jugaad/github/appliedmachinelearning/Assignment-0



                                                                                
100% Adding...|████████████████████████████████████████|3/3 [00:00, 20.38file/s]

To track the changes with git, run:

	git add data/validation.csv.dvc data/train.csv.dvc data/test.csv.dvc data/.gitignore

To enable auto staging, run:

	dvc config core.autostage true
